
## Notebook 04 - Feature Engineering

### Objectives

This notebook focuses on creating meaningful features from the cleaned dataset that can improve machine learning models and provide deeper analytical insights.

### Tasks

- Load the cleaned dataset
- Create datetime-based features
- Create pollution-based features
- Create weather-based features
- Create interaction features
- Create rolling and lag features
- Save the engineered dataset

In [ ]:
# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# File handling
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
# Project directory
PROJECT_DIR = Path("../")

# Data folders
DATA_DIR = PROJECT_DIR / "data"

# Input dataset
INPUT_PATH = DATA_DIR / "processed" / "clean_airintel.parquet"

# Output datasets
OUTPUT_PARQUET = DATA_DIR / "processed" / "feature_engineered_airintel.parquet"
OUTPUT_CSV = DATA_DIR / "processed" / "feature_engineered_airintel.csv"

In [ ]:
# Load cleaned dataset
df = pd.read_parquet(INPUT_PATH)

# Display dataset information
print(f"Dataset Shape : {df.shape}")

df.head()

In [ ]:
# Create working dataframe
df_feature = df.copy()

print("Working copy created successfully.")

## Create Datetime Features


In [ ]:
# Convert datetime column
df_feature["Datetime"] = pd.to_datetime(df_feature["Datetime"])

# Sort observations
df_feature = (
    df_feature
    .sort_values(["City", "Datetime"])
    .reset_index(drop=True)
)

In [ ]:
# Date 
df_feature["Date"] = df_feature["Datetime"].dt.date

In [ ]:
# Week

df_feature["Week"] = (
    df_feature["Datetime"]
    .dt.isocalendar()
    .week
    .astype(int)
)

In [ ]:
# Day of month
df_feature["Day_of_Month"] = df_feature["Datetime"].dt.day

In [ ]:
# Day of year
df_feature["Day_of_Year"] = df_feature["Datetime"].dt.dayofyear

In [ ]:
# Week of month
df_feature["Week_of_Month"] = (
    (df_feature["Datetime"].dt.day - 1) // 7
) + 1

In [ ]:
# Total number of days in the month
df_feature["Days_in_Month"] = (
    df_feature["Datetime"].dt.days_in_month
)

In [ ]:
# Check if the date is the first day of the month
df_feature["Is_Month_Start"] = (
    df_feature["Datetime"].dt.is_month_start.astype(int)
)

In [ ]:
# Check if the date is the last day of the month
df_feature["Is_Month_End"] = (
    df_feature["Datetime"].dt.is_month_end.astype(int)
)

In [ ]:
# Check if the date is the first day of a quarter
df_feature["Is_Quarter_Start"] = (
    df_feature["Datetime"].dt.is_quarter_start.astype(int)
)

In [ ]:
# Check if the date is the last day of a quarter
df_feature["Is_Quarter_End"] = (
    df_feature["Datetime"].dt.is_quarter_end.astype(int)
)

In [ ]:
new_datetime_features = [
    "Datetime",
    "Date",
    "Week",
    "Day_of_Month",
    "Day_of_Year",
    "Week_of_Month",
    "Days_in_Month",
    "Is_Month_Start",
    "Is_Month_End",
    "Is_Quarter_Start",
    "Is_Quarter_End"
]

df_feature[new_datetime_features].head()

In [ ]:
# Hour encoding
df_feature["Hour_Sin"] = np.sin(
    2 * np.pi * df_feature["Hour"] / 24
)

df_feature["Hour_Cos"] = np.cos(
    2 * np.pi * df_feature["Hour"] / 24
)

# Month encoding
df_feature["Month_Sin"] = np.sin(
    2 * np.pi * df_feature["Month"] / 12
)

df_feature["Month_Cos"] = np.cos(
    2 * np.pi * df_feature["Month"] / 12
)

# Weekday encoding
df_feature["Weekday_Sin"] = np.sin(
    2 * np.pi * df_feature["Day_of_Week"] / 7
)

df_feature["Weekday_Cos"] = np.cos(
    2 * np.pi * df_feature["Day_of_Week"] / 7
)

## Pollution Feature Engineering

In [ ]:
# List of pollutant columns
pollutant_cols = [
    "PM2_5_ugm3",
    "PM10_ugm3",
    "CO_ugm3",
    "NO2_ugm3",
    "SO2_ugm3",
    "O3_ugm3"
]

# Total pollution level
df_feature["Total_Pollution"] = df_feature[pollutant_cols].sum(axis=1)

# Mean Pollution 
df_feature["Mean_Pollution"] = (
    df_feature[pollutant_cols]
    .mean(axis=1)
)

In [ ]:
# Pollution Concentration
df_feature["Maximum_Pollution"] = (
    df_feature[pollutant_cols]
    .max(axis=1)
)

df_feature["Minimum_Pollution"] = (
    df_feature[pollutant_cols]
    .min(axis=1)
)

df_feature["Pollution_Range"] = (
    df_feature["Maximum_Pollution"]
    - df_feature["Minimum_Pollution"]
)

In [ ]:
# Number of available pollutant measurements
df_feature["Pollution_Count"] = (
    df_feature[pollutant_cols]
    .count(axis=1)
)

In [ ]:
# Difference between PM10 and PM2.5
df_feature["PM_Difference"] = (
    df_feature["PM10_ugm3"] -
    df_feature["PM2_5_ugm3"]
)

In [ ]:
# Ratio between CO and NO2
df_feature["CO_NO2_Ratio"] = (
    df_feature["CO_ugm3"] /
    (df_feature["NO2_ugm3"] + 1)
)

In [ ]:
# Ratio between PM2.5 and NO2
df_feature["PM25_NO2_Ratio"] = (
    df_feature["PM2_5_ugm3"] /
    (df_feature["NO2_ugm3"] + 1)
)

In [ ]:
# Ratio between O3 and NO2
df_feature["O3_NO2_Ratio"] = (
    df_feature["O3_ugm3"] /
    (df_feature["NO2_ugm3"] + 1)
)

In [ ]:
# Pollutant contributing the highest concentration
df_feature["Dominant_Pollutant"] = (
    df_feature[pollutant_cols]
    .idxmax(axis=1)
)

In [ ]:
new_pollution_features = [
    "Total_Pollution",
    "Mean_Pollution",
    "Maximum_Pollution",
    "Minimum_Pollution",
    "Pollution_Range",
    "Pollution_Count",
    "PM_Difference",
    "CO_NO2_Ratio",
    "PM25_NO2_Ratio",
    "O3_NO2_Ratio",
    "Dominant_Pollutant"
]

df_feature[new_pollution_features].head()

## Weather Feature Engineering

In [ ]:
# Difference between air temperature and dew point
df_feature["Temp_Dew_Diff"] = (
    df_feature["Temp_2m_C"] -
    df_feature["Dew_Point_C"]
)

In [ ]:
# Difference between sea-level and surface pressure
df_feature["Pressure_Difference"] = (
    df_feature["Pressure_MSL_hPa"] -
    df_feature["Surface_Pressure_hPa"]
)

In [ ]:
# Flag observations with strong winds
df_feature["High_Wind"] = (
    df_feature["Wind_Speed_10m_kmh"] > 20
).astype(int)

In [ ]:
# Flag observations with high cloud cover
df_feature["Heavy_Cloud"] = (
    df_feature["Cloud_Cover_Percent"] > 80
).astype(int)

In [ ]:
# Flag days with no rainfall
df_feature["Dry_Day"] = (
    df_feature["Rain_mm"] == 0
).astype(int)

In [ ]:
# Combined weather severity score
df_feature["Weather_Stress_Index"] = (
    df_feature["Wind_Speed_10m_kmh"] +
    df_feature["Humidity_Percent"] +
    df_feature["Cloud_Cover_Percent"]
)

In [ ]:
# Temperature × Humidity
df_feature["Temp_Humidity"] = (
    df_feature["Temp_2m_C"] *
    df_feature["Humidity_Percent"]
)

# Temperature × Wind Speed
df_feature["Temp_Wind"] = (
    df_feature["Temp_2m_C"] *
    df_feature["Wind_Speed_10m_kmh"]
)

# Rainfall × PM10
df_feature["Rain_PM10"] = (
    df_feature["Rain_mm"] *
    df_feature["PM10_ugm3"]
)

# Wind Speed × PM2.5
df_feature["Wind_PM25"] = (
    df_feature["Wind_Speed_10m_kmh"] *
    df_feature["PM2_5_ugm3"]
)

# Solar Radiation × Ozone
df_feature["Solar_O3"] = (
    df_feature["Solar_Radiation_Wm2"] *
    df_feature["O3_ugm3"]
)

# Humidity × PM2.5
df_feature["Humidity_PM25"] = (
    df_feature["Humidity_Percent"] *
    df_feature["PM2_5_ugm3"]
)

# Wind Gust Ratio
df_feature["Wind_Gust_Ratio"] = (
    df_feature["Wind_Gusts_kmh"] /
    (df_feature["Wind_Speed_10m_kmh"] + 1e-6)
)

In [ ]:
new_weather_features = [
    "Temp_Dew_Diff",
    "Pressure_Difference",
    "High_Wind",
    "Heavy_Cloud",
    "Dry_Day",
    "Weather_Stress_Index",
    "Temp_Humidity",
    "Temp_Wind",
    "Rain_PM10",
    "Wind_PM25",
    "Solar_O3",
    "Humidity_PM25",
    "Wind_Gust_Ratio"
]

df_feature[new_weather_features].head()

## Location Feature Engineering


In [ ]:
# Divide India into latitude bands
df_feature["Latitude_Band"] = pd.cut(
    df_feature["Latitude"],
    bins=5,
    labels=[
        "Very South",
        "South",
        "Central",
        "North",
        "Very North"
    ]
)

In [ ]:
# Divide India into longitude bands
df_feature["Longitude_Band"] = pd.cut(
    df_feature["Longitude"],
    bins=5,
    labels=[
        "West",
        "West-Central",
        "Central",
        "East-Central",
        "East"
    ]
)

In [ ]:
# List of coastal states and union territories
coastal_states = [
    "Gujarat",
    "Maharashtra",
    "Goa",
    "Karnataka",
    "Kerala",
    "Tamil Nadu",
    "Andhra Pradesh",
    "Odisha",
    "West Bengal",
    "Puducherry",
    "Andaman and Nicobar Islands",
    "Lakshadweep",
    "Daman and Diu"
]

# Flag coastal locations
df_feature["Coastal_State"] = (
    df_feature["State"]
    .isin(coastal_states)
    .astype(int)
)

In [ ]:
# States considered part of Northern India
north_states = [
    "Delhi",
    "Punjab",
    "Haryana",
    "Himachal Pradesh",
    "Jammu and Kashmir",
    "Ladakh",
    "Uttarakhand",
    "Uttar Pradesh",
    "Chandigarh"
]

# Flag northern locations
df_feature["Northern_India"] = (
    df_feature["State"]
    .isin(north_states)
    .astype(int)
)

In [ ]:
# Flag non-coastal locations
df_feature["Inland_State"] = (
    1 - df_feature["Coastal_State"]
)

In [ ]:
# Absolute latitude
df_feature["Absolute_Latitude"] = (
    df_feature["Latitude"].abs()
)

# Latitude × Longitude interaction
df_feature["Lat_Long_Interaction"] = (
    df_feature["Latitude"] *
    df_feature["Longitude"]
)

In [ ]:
location_features = [
    "Latitude_Band",
    "Longitude_Band",
    "Coastal_State",
    "Northern_India",
    "Inland_State",
    "Absolute_Latitude",
    "Lat_Long_Interaction"
]

df_feature[location_features].head()

## Time-Series Feature Engineering


In [ ]:
# Lag intervals (hours)
lag_hours = [1, 3, 6, 12, 24]

# Ensure proper ordering
df_feature = (
    df_feature
    .sort_values(["City", "Datetime"])
    .reset_index(drop=True)
)

In [ ]:
# Create lag features for each pollutant

for pollutant in pollutant_cols:

    for lag in lag_hours:

        df_feature[f"{pollutant}_Lag_{lag}H"] = (

            df_feature
            .groupby("City")[pollutant]
            .shift(lag)

        )

In [ ]:
# Rolling window sizes (hours)
rolling_windows = [6, 12, 24]

for pollutant in pollutant_cols:

    grouped = df_feature.groupby("City")[pollutant]

    for window in rolling_windows:

        df_feature[f"{pollutant}_RollingMean_{window}H"] = (

            grouped

            .transform(

                lambda x:
                x.rolling(window, min_periods=1).mean()

            )

        )

        df_feature[f"{pollutant}_RollingStd_{window}H"] = (

            grouped

            .transform(

                lambda x:
                x.rolling(window, min_periods=1).std(ddof=0)
            )

        )

In [ ]:
# Create rolling minimum and maximum

for pollutant in pollutant_cols:

    grouped = df_feature.groupby("City")[pollutant]

    for window in rolling_windows:

        df_feature[f"{pollutant}_RollingMin_{window}H"] = (

            grouped.transform(

                lambda x:
                x.rolling(window, min_periods=1).min()

            )

        )

        df_feature[f"{pollutant}_RollingMax_{window}H"] = (

            grouped.transform(

                lambda x:
                x.rolling(window, min_periods=1).max()

            )

        )

In [ ]:
### Exponential Moving Average
ema_windows = [6, 12, 24]

for pollutant in pollutant_cols:

    grouped = df_feature.groupby("City")[pollutant]

    for window in ema_windows:

        df_feature[f"{pollutant}_EMA_{window}H"] = (

            grouped

            .transform(

                lambda x:
                x.ewm(
                    span=window,
                    adjust=False
                ).mean()

            )

        )

In [ ]:
# Hourly pollutant change - Trend Features


for pollutant in pollutant_cols:

    df_feature[f"{pollutant}_Change"] = (

        df_feature
        .groupby("City")[pollutant]
        .diff()

    )

    df_feature[f"{pollutant}_PctChange"] = (

        df_feature
        .groupby("City")[pollutant]
        .pct_change()

    )

In [ ]:
# Pollution volatility

for pollutant in pollutant_cols:

    df_feature[f"{pollutant}_Volatility_24H"] = (

        df_feature
        .groupby("City")[pollutant]

        .transform(

            lambda x:
            x.rolling(24, min_periods=2).std(ddof=0)

        )

    )

In [ ]:
# Display sample engineered time-series features

time_series_features = [

    "PM2_5_ugm3_Lag_1H",
    "PM2_5_ugm3_Lag_24H",
    "PM2_5_ugm3_RollingMean_24H",
    "PM2_5_ugm3_RollingStd_24H",
    "PM2_5_ugm3_EMA_24H",
    "PM2_5_ugm3_Change",
    "PM2_5_ugm3_PctChange"

]

df_feature[time_series_features].head(10)

## Validate Engineered Features


In [ ]:
# Dataset overview
print(f"Dataset Shape : {df_feature.shape}")

print("\nNumerical Features :", len(df_feature.select_dtypes(include=np.number).columns))
print("Categorical Features :", len(df_feature.select_dtypes(exclude=np.number).columns))

In [ ]:
# Check missing values

missing_features = pd.DataFrame({

    "Missing Values": df_feature.isna().sum(),
    "Missing %": (
        df_feature.isna().mean() * 100
    ).round(2)

})

missing_features = missing_features[
    missing_features["Missing Values"] > 0
].sort_values(
    "Missing Values",
    ascending=False
)

missing_features.head(20)

## Feature Engineering Summary

In [ ]:
# Summarize feature engineering

feature_summary = pd.DataFrame({

    "Metric": [

        "Rows",
        "Original Columns",
        "Final Columns",
        "New Features Added",
        "Numerical Features",
        "Categorical Features",
        "Memory Usage (MB)"

    ],

    "Value": [

        len(df_feature),

        df.shape[1],

        df_feature.shape[1],

        df_feature.shape[1] - df.shape[1],

        len(
            df_feature.select_dtypes(
                include=np.number
            ).columns
        ),

        len(
            df_feature.select_dtypes(
                exclude=np.number
            ).columns
        ),

        round(
            df_feature.memory_usage(deep=True).sum() / 1024**2,
            2
        )

    ]

})

feature_summary

## Feature Engineering Decisions


In [ ]:
# Document feature engineering decisions

feature_decisions = pd.DataFrame({

    "Feature Group": [

        "Datetime",
        "Cyclical",
        "Pollution",
        "Weather",
        "Weather Interaction",
        "Spatial",
        "Time-Series"

    ],

    "Purpose": [

        "Calendar-based patterns",
        "Preserve cyclic behaviour",
        "Pollution relationships",
        "Meteorological conditions",
        "Pollution-weather effects",
        "Geographical information",
        "Temporal dependencies"

    ]

})

feature_decisions

## Save the Feature Engineered Dataset

In [ ]:
# Save feature-engineered dataset

df_feature.to_parquet(
    OUTPUT_PARQUET,
    index=False
)

df_feature.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Feature engineered dataset saved successfully.")
print(f"Rows    : {len(df_feature):,}")
print(f"Columns : {df_feature.shape[1]}")

## Conclusion

The cleaned dataset has been successfully transformed into a feature-rich analytical dataset by incorporating temporal, pollution, weather, spatial and time-series features.

The resulting dataset is now ready for advanced exploratory data analysis, statistical analysis and predictive modeling.